# 11_rebuild_gold
**It runs 10_silver_to_gold once per loaded date, oldest first, so every fact gets its history. It takes roughly 15 to 25 minutes.**


In [0]:
history = spark.table("workspace.steam.silver_steam_history")
dates = [str(r.load_date) for r in history.select("load_date").distinct().orderBy("load_date").collect()]
print("dates:", dates)

for d in dates:
    dbutils.notebook.run("./10_silver_to_gold", 3600, {"date": d})
    n = spark.sql(f"SELECT COUNT(*) FROM workspace.gold.fact_game_daily_snapshot "
                  f"WHERE date_key = {int(d.replace('-', ''))}").first()[0]
    print(d, "done | snapshot rows:", n)

In [0]:
display(spark.sql("""
    SELECT file_processed, status, rows_inserted, rows_updated, start_time
    FROM workspace.steam.pipeline_execution_logs
    WHERE layer = 'Silver-to-Gold'
    ORDER BY start_time DESC LIMIT 3
"""))